[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadErlanggaKurniawan/ets-ml2-kelompok/blob/main/notebooks/02_track_B_kelarutan_aqsoldb.ipynb)

# 02 · Track B — Kelarutan Air Senyawa Obat (AqSolDB)

Track ini sesuai dengan topik skripsi Michael: gradient tree-boosting, Optuna, dan SHAP pada AqSolDB
(9.982 senyawa), dengan **scaffold-based splitting**.

| Aspek | Bagian |
|---|---|
| Wrangling | §1 deduplikasi, scaffold Bemis–Murcko, 207 deskriptor RDKit-2D |
| Validasi | §2 split acak vs scaffold |
| Ensemble | §3 model tunggal vs RF/XGBoost/LightGBM/Stacking |
| Optimasi Bayesian | §4 Optuna (TPE) dengan CV berkelompok scaffold |
| Imbalanced regression | §5 bobot frekuensi-invers per kategori kelarutan |
| Interpretabilitas | §6 SHAP |

## 0. Instalasi & utilitas

In [ ]:
# Versi dikunci sama dengan uv.lock di repo supaya angka di Colab = angka di laporan
%pip install -q scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 optuna==5.0.0 \
    imbalanced-learn==0.14.2 rdkit==2026.9.1 shap==0.54.1

In [ ]:
import warnings, hashlib, json, time, urllib.request, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.base import clone
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              StackingClassifier, StackingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", message="The `probability` parameter was deprecated")
SEED = 42
RAW, RESULTS = Path("data/raw"), Path("results")
RAW.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch(url, dest):
    import ssl, certifi                       # certifi: CA bundle yang sama di Colab dan macOS
    dest = Path(dest)
    if not dest.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "ets-ml2"})
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(req, context=ctx) as r, open(dest, "wb") as f:
            f.write(r.read())
    return dest

def mean_sd(v):
    v = np.asarray(v, float)
    return f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}"

## 1. Akuisisi & wrangling

In [ ]:
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
RDLogger.DisableLog("rdApp.*")

# AqSolDB (Sorkun dkk., 2019, Sci. Data 6:143) - mirror Hugging Face maomlab/AqSolDB (MIT)
SHA = {"train.csv": "b9572e57235f1cfb826f899d640e71617c1ad6014c3e38450c21282ca1962463",
       "test.csv": "e20826b231945ea9e02a7e515c0f56e2f0f98b9c97110c2ed49911bf21550f32"}
parts = []
for f, h in SHA.items():
    p = fetch(f"https://huggingface.co/datasets/maomlab/AqSolDB/resolve/main/AqSolDB/{f}", RAW / f"aqsoldb_{f}")
    assert sha256(p) == h, f"checksum {f} tidak cocok"
    parts.append(pd.read_csv(p))
df = pd.concat(parts, ignore_index=True).drop_duplicates("InChIKey").rename(columns={"Y": "logS"}).reset_index(drop=True)
BINS = [-np.inf, -6, -4, -2, 0, np.inf]
BIN_LABELS = ["very_insoluble(<-6)", "insoluble(-6..-4)", "slightly(-4..-2)", "soluble(-2..0)", "highly(>=0)"]
df["sol_bin"] = pd.cut(df["logS"], BINS, labels=BIN_LABELS, right=False)

def scaffold_of(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return f"INVALID:{smi}"
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except Exception: return f"INVALID:{smi}"
df["scaffold"] = df["SMILES"].map(scaffold_of)
empty = df["scaffold"] == ""                     # molekul asiklik: masing-masing grup sendiri
df.loc[empty, "scaffold"] = "ACYCLIC:" + df.loc[empty, "InChIKey"]

rows = []
for smi in df["SMILES"]:
    mol = Chem.MolFromSmiles(smi)
    rows.append([fn(mol) if mol is not None else np.nan for _, fn in Descriptors.descList])
X_rd = pd.DataFrame(rows, columns=[n for n, _ in Descriptors.descList]).replace([np.inf, -np.inf], np.nan)
X_rd = X_rd.loc[:, X_rd.isna().mean() < 0.01].clip(-1e6, 1e6)
X_rd = X_rd.loc[:, X_rd.std() > 0].astype("float32")
AQSOL_DESC = ["MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors", "NumHeteroatoms",
              "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings", "NumSaturatedRings",
              "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA", "BalabanJ", "BertzCT"]
X_aq = df[AQSOL_DESC].astype("float32"); y = df["logS"].to_numpy()
print(f"n={len(df)}  scaffold unik={df.scaffold.nunique()}  fitur RDKit-2D={X_rd.shape[1]}")
df.sol_bin.value_counts().sort_index()

## 2. Scaffold split vs split acak
Molekul dengan scaffold yang sama mirip bagi model. Split acak membocorkan "kimia" dari data latih ke
data uji. Pada scaffold split, grup scaffold besar masuk ke data latih dan scaffold langka mengisi data uji,
sehingga data uji berisi **kimia baru**. Ini skenario yang paling sulit sekaligus paling realistis.

In [ ]:
def scaffold_split(d, test_frac=0.2, seed=SEED):
    groups = defaultdict(list)
    for i, s in enumerate(d["scaffold"]): groups[s].append(i)
    keys = list(groups); np.random.default_rng(seed).shuffle(keys)
    keys.sort(key=lambda k: len(groups[k]), reverse=True)
    n_train, train, test = int(round((1 - test_frac) * len(d))), [], []
    for k in keys:
        (train if len(train) + len(groups[k]) <= n_train else test).extend(groups[k])
    return np.array(sorted(train)), np.array(sorted(test))

tr, te = scaffold_split(df)
perm = np.random.default_rng(SEED).permutation(len(df)); k = int(round(.2 * len(df)))
rtr, rte = np.sort(perm[k:]), np.sort(perm[:k])
print("scaffold uji yang juga ada di latih:", df.scaffold.iloc[te].isin(df.scaffold.iloc[tr]).sum())
inner = list(GroupKFold(5).split(tr, groups=df["scaffold"].iloc[tr].to_numpy()))

def reg_scores(t, p):
    return {"rmse": float(np.sqrt(mean_squared_error(t, p))), "mae": float(mean_absolute_error(t, p)),
            "r2": float(r2_score(t, p))}

SUMMARY = []
def run(tag, name, model, X, split="scaffold", label="", sample_weight=None):
    a, b = (tr, te) if split == "scaffold" else (rtr, rte)
    m = clone(model)
    if hasattr(m, "cv") and split == "scaffold": m.set_params(cv=inner)
    kw = {"sample_weight": sample_weight} if sample_weight is not None else {}
    p = m.fit(X.iloc[a], y[a], **kw).predict(X.iloc[b]); s = reg_scores(y[b], p)
    SUMMARY.append({"experiment": tag, "model": name, "split": split, "variant": label, **s})
    print(f"{tag:12s} {name:16s} {split:8s} {label:18s} RMSE={s['rmse']:.3f}  R2={s['r2']:.3f}")
    return m, p

imp = lambda: SimpleImputer(strategy="median")
SINGLE = {"Ridge": make_pipeline(imp(), StandardScaler(), Ridge(alpha=1.0)),
          "KNN": make_pipeline(imp(), StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
          "SVR-RBF": make_pipeline(imp(), StandardScaler(), SVR(C=10, gamma="scale")),
          "DecisionTree": DecisionTreeRegressor(max_depth=10, random_state=SEED)}
ENSEMBLE = {"RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
            "XGBoost": XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                                    colsample_bytree=0.8, n_jobs=4, random_state=SEED, verbosity=0),
            "LightGBM": lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                                          random_state=SEED, verbose=-1, n_jobs=4),
            "Stacking": StackingRegressor(
                estimators=[("ridge", make_pipeline(imp(), StandardScaler(), Ridge(alpha=1.0))),
                            ("svr", make_pipeline(imp(), StandardScaler(), SVR(C=10, gamma="scale"))),
                            ("rf", RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1,
                                                         random_state=SEED)),
                            ("lgbm", lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05,
                                                       random_state=SEED, verbose=-1, n_jobs=4))],
                final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)}

for split in ["random", "scaffold"]:
    for fname, X in [("AqSolDB-17", X_aq), ("RDKit-2D", X_rd)]:
        run("S1_split", "LightGBM", ENSEMBLE["LightGBM"], X, split, fname)
        run("S1_split", "RandomForest", ENSEMBLE["RandomForest"], X, split, fname)

## 3. Ensemble learning — model tunggal vs ensemble (scaffold split, RDKit-2D)

In [ ]:
preds = {}
for name, m in {**SINGLE, **ENSEMBLE}.items():
    _, preds[name] = run("S2_models", name, m, X_rd, label="RDKit-2D")
s = pd.DataFrame(SUMMARY).query("experiment == 'S2_models'")
s.plot.barh(x="model", y="rmse", legend=False, title="RMSE uji scaffold (lebih kecil lebih baik)",
            color=["#9aa5a6"] * 4 + ["#0f5257"] * 4, figsize=(7, 4)); plt.show()

## 4. Optimasi Bayesian dengan Optuna (TPE)
CV di dalam data latih dikelompokkan per scaffold. Data uji tidak disentuh selama tuning.
`N_TRIALS = 40` sama dengan laporan; turunkan kalau waktu Colab terbatas.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 40
Xtr, ytr = X_rd.iloc[tr], y[tr]
def objective(trial):
    params = dict(n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=100),
                  learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                  num_leaves=trial.suggest_int("num_leaves", 15, 127),
                  min_child_samples=trial.suggest_int("min_child_samples", 5, 60),
                  subsample=trial.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
                  colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
                  reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
                  random_state=SEED, verbose=-1, n_jobs=4)
    errs = [np.sqrt(np.mean((lgb.LGBMRegressor(**params).fit(Xtr.iloc[a], ytr[a]).predict(Xtr.iloc[b]) - ytr[b]) ** 2))
            for a, b in inner]
    return float(np.mean(errs))
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("CV RMSE terbaik:", round(study.best_value, 3), study.best_params)
best = lgb.LGBMRegressor(**study.best_params, subsample_freq=1, random_state=SEED, verbose=-1, n_jobs=4)
tuned, p_tuned = run("S3_optuna", "LightGBM-Optuna", best, X_rd, label=f"{N_TRIALS} trials TPE")

**Bacaan:** CV RMSE hasil tuning (≈1,06) jauh lebih baik daripada RMSE pada data uji (≈1,26).
Data uji berisi scaffold langka, jadi distribusinya bergeser dari fold CV. Tuning tidak bisa menutup
*distribution shift*.

## 5. Imbalanced regression — bobot frekuensi-invers per kategori kelarutan

In [ ]:
bins_tr = df["sol_bin"].iloc[tr].astype(str)
w = 1.0 / bins_tr.map(bins_tr.value_counts(normalize=True)).astype(float).to_numpy(); w = w / w.mean()
_, p_w = run("S4_imbalance", "LightGBM-Optuna", best, X_rd, label="inv-freq weights", sample_weight=w)
t = df.iloc[te].assign(err_u=np.abs(y[te] - p_tuned), err_w=np.abs(y[te] - p_w))
per_bin = t.groupby("sol_bin", observed=False).agg(n=("err_u", "size"), mae_unweighted=("err_u", "mean"),
                                                   mae_weighted=("err_w", "mean"))
display(per_bin.round(3)); per_bin[["mae_unweighted", "mae_weighted"]].plot.bar(title="MAE per kategori"); plt.show()
pd.DataFrame(SUMMARY).to_csv(RESULTS / "solubility_summary.csv", index=False)

## 6. Interpretabilitas — SHAP pada model hasil tuning

In [ ]:
import shap
sample = X_rd.iloc[te].sample(1000, random_state=SEED)
sv = shap.TreeExplainer(tuned).shap_values(sample)
shap.summary_plot(sv, sample, max_display=12, show=True)